In [2]:
# Downgrade to a compatible 1.x version of numpy for scikit-surprise
!pip install "numpy<2" "scikit-surprise"


In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print(f"Numpy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")

Numpy version: 1.26.4
Pandas version: 2.2.2


In [4]:
!wget https://files.grouplens.org/datasets/movielens/ml-100k.zip
!unzip ml-100k.zip

--2026-05-18 17:22:03--  https://files.grouplens.org/datasets/movielens/ml-100k.zip
Resolving files.grouplens.org (files.grouplens.org)... 128.101.96.204
Connecting to files.grouplens.org (files.grouplens.org)|128.101.96.204|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 4924029 (4.7M) [application/zip]
Saving to: ‘ml-100k.zip.1’

ml-100k.zip.1       100%[===================>]   4.70M  2.75MB/s    in 1.7s    

2026-05-18 17:22:06 (2.75 MB/s) - ‘ml-100k.zip.1’ saved [4924029/4924029]

Archive:  ml-100k.zip
replace ml-100k/allbut.pl? [y]es, [n]o, [A]ll, [N]one, [r]ename: A
  inflating: ml-100k/allbut.pl       
  inflating: ml-100k/mku.sh          
  inflating: ml-100k/README          
  inflating: ml-100k/u.data          
  inflating: ml-100k/u.genre         
  inflating: ml-100k/u.info          
  inflating: ml-100k/u.item          
  inflating: ml-100k/u.occupation    
  inflating: ml-100k/u.user          
  inflating: ml-100k/u1.base         
  inflating: ml

In [5]:
ratings = pd.read_csv(
    "ml-100k/u.data",
    sep="\t",
    names=["user_id", "movie_id", "rating", "timestamp"]
)

movie_columns = [
    "movie_id", "title", "release_date", "video_release_date",
    "IMDb_URL", "unknown", "Action", "Adventure", "Animation",
    "Children", "Comedy", "Crime", "Documentary", "Drama",
    "Fantasy", "FilmNoir", "Horror", "Musical", "Mystery",
    "Romance", "SciFi", "Thriller", "War", "Western"
]

movies = pd.read_csv(
    "ml-100k/u.item",
    sep="|",
    encoding="latin-1",
    header=None,
    names = movie_columns,
)

users = pd.read_csv(
    "ml-100k/u.user",
    sep="|",
    names=["user_id", "age", "gender", "occupation", "zip_code"]
)

In [6]:
print(f"Ratings NA count: {ratings.isna().sum().sum()}")
print(f"Movies NA count: {movies.isna().sum().sum()}")
print(f"Users NA count: {users.isna().sum().sum()}")

print(f"Ratings Duplicates count: {ratings.duplicated().sum().sum()}")
print(f"Movies Duplicates count: {movies.duplicated().sum().sum()}")
print(f"Users Duplicates count: {users.duplicated().sum().sum()}")

Ratings NA count: 0
Movies NA count: 1686
Users NA count: 0
Ratings Duplicates count: 0
Movies Duplicates count: 0
Users Duplicates count: 0


In [7]:
movies = movies.drop(columns=["video_release_date"]) # empty column

In [8]:
movies

,movie_id,title,release_date,IMDb_URL,unknown,Action,Adventure,Animation,Children,Comedy,...,Fantasy,FilmNoir,Horror,Musical,Mystery,Romance,SciFi,Thriller,War,Western
0,1,Toy Story (1995),01-Jan-1995,http://us.imdb.com/M/title-exact?Toy%20Story%2...,0,0,0,1,1,1,...,0,0,0,0,0,0,0,0,0,0
1,2,GoldenEye (1995),01-Jan-1995,http://us.imdb.com/M/title-exact?GoldenEye%20(...,0,1,1,0,0,0,...,0,0,0,0,0,0,0,1,0,0
2,3,Four Rooms (1995),01-Jan-1995,http://us.imdb.com/M/title-exact?Four%20Rooms%...,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
3,4,Get Shorty (1995),01-Jan-1995,http://us.imdb.com/M/title-exact?Get%20Shorty%...,0,1,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
4,5,Copycat (1995),01-Jan-1995,http://us.imdb.com/M/title-exact?Copycat%20(1995),0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1677,1678,Mat' i syn (1997),06-Feb-1998,http://us.imdb.com/M/title-exact?Mat%27+i+syn+...,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1678,1679,B. Monkey (1998),06-Feb-1998,http://us.imdb.com/M/title-exact?B%2E+Monkey+(...,0,0,0,0,0,0,...,0,0,0,0,0,1,0,1,0,0
1679,1680,Sliding Doors (1998),01-Jan-1998,http://us.imdb.com/Title?Sliding+Doors+(1998),0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
1680,1681,You So Crazy (1994),01-Jan-1994,http://us.imdb.com/M/title-exact?You%20So%20Cr...,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0


In [9]:
movies["release_date"] = pd.to_datetime(movies["release_date"])
movies

,movie_id,title,release_date,IMDb_URL,unknown,Action,Adventure,Animation,Children,Comedy,...,Fantasy,FilmNoir,Horror,Musical,Mystery,Romance,SciFi,Thriller,War,Western
0,1,Toy Story (1995),1995-01-01,http://us.imdb.com/M/title-exact?Toy%20Story%2...,0,0,0,1,1,1,...,0,0,0,0,0,0,0,0,0,0
1,2,GoldenEye (1995),1995-01-01,http://us.imdb.com/M/title-exact?GoldenEye%20(...,0,1,1,0,0,0,...,0,0,0,0,0,0,0,1,0,0
2,3,Four Rooms (1995),1995-01-01,http://us.imdb.com/M/title-exact?Four%20Rooms%...,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
3,4,Get Shorty (1995),1995-01-01,http://us.imdb.com/M/title-exact?Get%20Shorty%...,0,1,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
4,5,Copycat (1995),1995-01-01,http://us.imdb.com/M/title-exact?Copycat%20(1995),0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1677,1678,Mat' i syn (1997),1998-02-06,http://us.imdb.com/M/title-exact?Mat%27+i+syn+...,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1678,1679,B. Monkey (1998),1998-02-06,http://us.imdb.com/M/title-exact?B%2E+Monkey+(...,0,0,0,0,0,0,...,0,0,0,0,0,1,0,1,0,0
1679,1680,Sliding Doors (1998),1998-01-01,http://us.imdb.com/Title?Sliding+Doors+(1998),0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
1680,1681,You So Crazy (1994),1994-01-01,http://us.imdb.com/M/title-exact?You%20So%20Cr...,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0


In [10]:
movies = movies.dropna() # only 4

In [11]:
# movies.drop(columns=["movie_id"]).boxplot() # outliers stuff
# plt.figure(figsize=(20, 20))
# plt.tight_layout()
# plt.show()

In [12]:
print(f"Count of movies with unknown genre: {len(movies[movies["unknown"] != 0])}")
movies = movies.drop(columns=["unknown"])
movies

Count of movies with unknown genre: 1


,movie_id,title,release_date,IMDb_URL,Action,Adventure,Animation,Children,Comedy,Crime,...,Fantasy,FilmNoir,Horror,Musical,Mystery,Romance,SciFi,Thriller,War,Western
0,1,Toy Story (1995),1995-01-01,http://us.imdb.com/M/title-exact?Toy%20Story%2...,0,0,1,1,1,0,...,0,0,0,0,0,0,0,0,0,0
1,2,GoldenEye (1995),1995-01-01,http://us.imdb.com/M/title-exact?GoldenEye%20(...,1,1,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
2,3,Four Rooms (1995),1995-01-01,http://us.imdb.com/M/title-exact?Four%20Rooms%...,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
3,4,Get Shorty (1995),1995-01-01,http://us.imdb.com/M/title-exact?Get%20Shorty%...,1,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
4,5,Copycat (1995),1995-01-01,http://us.imdb.com/M/title-exact?Copycat%20(1995),0,0,0,0,0,1,...,0,0,0,0,0,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1677,1678,Mat' i syn (1997),1998-02-06,http://us.imdb.com/M/title-exact?Mat%27+i+syn+...,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1678,1679,B. Monkey (1998),1998-02-06,http://us.imdb.com/M/title-exact?B%2E+Monkey+(...,0,0,0,0,0,0,...,0,0,0,0,0,1,0,1,0,0
1679,1680,Sliding Doors (1998),1998-01-01,http://us.imdb.com/Title?Sliding+Doors+(1998),0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
1680,1681,You So Crazy (1994),1994-01-01,http://us.imdb.com/M/title-exact?You%20So%20Cr...,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0


In [13]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

genres = ["Action", "Adventure", "Animation",
    "Children", "Comedy", "Crime", "Documentary", "Drama",
    "Fantasy", "FilmNoir", "Horror", "Musical", "Mystery",
    "Romance", "SciFi", "Thriller", "War", "Western"]

vectorizer = TfidfVectorizer()
tfidfmatrix = vectorizer.fit_transform(movies[genres])

# movie example
shawerma_movie = [1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

similarities = cosine_similarity(tfidfmatrix, [shawerma_movie])

highest_similarity_idx = np.argmax(similarities)

most_similar_movie = movies.iloc[highest_similarity_idx]["title"]

most_similar_movie

'Toy Story (1995)'

In [14]:
from surprise import SVD, Reader, Dataset
from surprise.model_selection import cross_validate
from surprise.model_selection import train_test_split
from surprise.accuracy import mae, rmse

reader = Reader(rating_scale=(1, 5))

data = Dataset.load_from_df(
    ratings[["user_id", "movie_id", "rating"]],
    reader
)

train_set, test_set = train_test_split(data, test_size = 0.2, random_state= 42)

svd = SVD()
svd.fit(train_set)

predictions = svd.test(test_set)

mae(predictions, test_set)
rmse(predictions, test_set)

MAE:  0.7379
RMSE: 0.9382


0.9381618255856652

In [15]:
# example of ranking movies based on predicted user rating, tfidf gets similar movies and then they are ranked by their expected rating

user_id = 2

movie_ids = ratings[(ratings["user_id"] == user_id) & (ratings["rating"] >=4)]["movie_id"].values

liked_movies = movies[movies["movie_id"].isin(movie_ids)]

similarities = cosine_similarity(liked_movies[genres], movies[genres])


test_set = [(user_id, movie_id, 0) for movie_id in movies["movie_id"]]

expected_ratings = svd.test(test_set)
expected_ratings_pred = [expected_ratings[i].est for i in range(len(expected_ratings))]
expected_ratings_pred = np.array(expected_ratings_pred)

# similarities has similarities compared to all movies, expected_ratings has expected_rating for all movies

# print(similarities)

scores = 0.3 * np.sum(similarities, axis=0) + 0.7 * expected_ratings_pred

df_dict = {"movie_id": movies["movie_id"],
           "movie_name": movies["title"],
           "expected rating": expected_ratings_pred,
           "score": scores}

result_df = pd.DataFrame(df_dict)
result_df = result_df[~result_df["movie_id"].isin(movie_ids)] # remove already watched movies
result_df.sort_values(by="score", ascending=False).head(10)

,movie_id,movie_name,expected rating,score
169,170,Cinema Paradiso (1988),4.207459,9.279324
516,517,Manhattan (1979),4.158527,9.245072
511,512,Wings of Desire (1987),4.092266,9.198689
935,936,Brassed Off (1996),3.952495,9.100849
777,778,Don Juan DeMarco (1995),3.831651,9.016258
480,481,"Apartment, The (1960)",4.174056,8.867207
44,45,Eat Drink Man Woman (1994),4.146300,8.847778
149,150,Swingers (1996),4.103052,8.817504
522,523,Cool Hand Luke (1967),4.086858,8.806169
1114,1115,Twelfth Night (1996),3.528932,8.804355


## Evaluation: Precision, Recall and F1-Score

In [16]:
from collections import defaultdict

def precision_recall_at_k(predictions, k=10, threshold=3.5):
    '''Return precision and recall at k metrics for each user.'''

    # First map the predictions to each user.
    user_est_true = defaultdict(list)
    for uid, _, true_r, est, _ in predictions:
        user_est_true[uid].append((est, true_r))

    precisions = dict()
    recalls = dict()
    for uid, user_ratings in user_est_true.items():
        # Sort user ratings by estimated value
        user_ratings.sort(key=lambda x: x[0], reverse=True)

        # Number of relevant items
        n_rel = sum((true_r >= threshold) for (_, true_r) in user_ratings)

        # Number of recommended items in top k
        n_rec_k = sum((est >= threshold) for (est, _) in user_ratings[:k])

        # Number of relevant and recommended items in top k
        n_rel_and_rec_k = sum(((true_r >= threshold) and (est >= threshold))
                              for (est, true_r) in user_ratings[:k])

        # Precision@K: Proportion of recommended items that are relevant
        precisions[uid] = n_rel_and_rec_k / n_rec_k if n_rec_k != 0 else 1

        # Recall@K: Proportion of relevant items that are recommended
        recalls[uid] = n_rel_and_rec_k / n_rel if n_rel != 0 else 1

    return precisions, recalls

precisions, recalls = precision_recall_at_k(predictions, k=10, threshold=4)

mean_precision = sum(prec for prec in precisions.values()) / len(precisions)
mean_recall = sum(rec for rec in recalls.values()) / len(recalls)
f1_score = 2 * (mean_precision * mean_recall) / (mean_precision + mean_recall)

print(f"Precision@10: {mean_precision:.4f}")
print(f"Recall@10: {mean_recall:.4f}")
print(f"F1-Score@10: {f1_score:.4f}")


Precision@10: 0.8737
Recall@10: 0.3045
F1-Score@10: 0.4516


## Model Comparison (CF vs CB vs Hybrid)

In [19]:
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("Evaluating all models to compare performance...")

# Correct TF-IDF Implementation for Evaluation
movies['genres_str'] = movies[genres].apply(lambda row: ' '.join(row.index[row == 1]), axis=1)
tfidf = TfidfVectorizer()
tfidf_matrix = tfidf.fit_transform(movies['genres_str'])

# Build full item similarity matrix using TF-IDF
item_sim_matrix = cosine_similarity(tfidf_matrix, tfidf_matrix)
movie_id_to_idx = {mid: i for i, mid in enumerate(movies['movie_id'])}

train_raw_ratings = train_set.build_testset()
train_dict = {}
for u, i, r in train_raw_ratings:
    if u not in train_dict:
        train_dict[u] = []
    train_dict[u].append((i, r))

true_ratings = []
cf_preds = []
cb_preds = []
hybrid_preds = []

w_cf = 0.7
w_cb = 0.3

for u, i, r in test_set:
    true_ratings.append(r)

    cf_est = svd.predict(u, i).est
    cf_preds.append(cf_est)

    cb_est = 3.0 # default
    if u in train_dict and i in movie_id_to_idx:
        u_ratings = train_dict[u]
        target_idx = movie_id_to_idx[i]

        sim_sum = 0
        weighted_sum = 0
        for past_i, past_r in u_ratings:
            if past_i in movie_id_to_idx:
                past_idx = movie_id_to_idx[past_i]
                sim = item_sim_matrix[target_idx, past_idx]
                sim_sum += sim
                weighted_sum += sim * past_r

        if sim_sum > 0:
            cb_est = weighted_sum / sim_sum
        else:
            cb_est = np.mean([r for _, r in u_ratings])

    cb_preds.append(cb_est)
    hybrid_preds.append(w_cf * cf_est + w_cb * cb_est)

print("\n--- Model Evaluation Comparison ---")
print(f"RMSE Collaborative Filtering: {np.sqrt(mean_squared_error(true_ratings, cf_preds)):.4f}")
print(f"RMSE Content-Based Filtering: {np.sqrt(mean_squared_error(true_ratings, cb_preds)):.4f}")
print(f"RMSE Hybrid Engine:           {np.sqrt(mean_squared_error(true_ratings, hybrid_preds)):.4f}")

print(f"\nMAE Collaborative Filtering: {mean_absolute_error(true_ratings, cf_preds):.4f}")
print(f"MAE Content-Based Filtering: {mean_absolute_error(true_ratings, cb_preds):.4f}")
print(f"MAE Hybrid Engine:           {mean_absolute_error(true_ratings, hybrid_preds):.4f}")


Evaluating all models to compare performance...

--- Model Evaluation Comparison ---
RMSE Collaborative Filtering: 3.4575
RMSE Content-Based Filtering: 3.7074
RMSE Hybrid Engine:           3.5266

MAE Collaborative Filtering: 3.4305
MAE Content-Based Filtering: 3.6997
MAE Hybrid Engine:           3.5112


## Hybrid Recommendation Engine Function

In [21]:
def hybrid_recommendation(user_id, movies_df, ratings_df, svd_model, genres_list, weight_cb=0.3, weight_cf=0.7, top_n=10):
    """
    Generates hybrid recommendations combining Content-Based and Collaborative Filtering.
    Uses TF-IDF on genres.
    """
    import pandas as pd
    import numpy as np
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.metrics.pairwise import cosine_similarity

    # 1. Correct TF-IDF Implementation for Content-Based Filtering
    # Convert binary genre features into a string of genre names for TF-IDF
    movies_df['genres_str'] = movies_df[genres_list].apply(lambda row: ' '.join(row.index[row == 1]), axis=1)
    tfidf = TfidfVectorizer(stop_words='english')
    tfidf_matrix = tfidf.fit_transform(movies_df['genres_str'])

    user_ratings = ratings_df[(ratings_df["user_id"] == user_id) & (ratings_df["rating"] >= 4)]

    if len(user_ratings) == 0:
        user_ratings = ratings_df[ratings_df["user_id"] == user_id]

    movie_ids = user_ratings["movie_id"].values

    if len(movie_ids) == 0:
        similarities = np.zeros(len(movies_df))
    else:
        # Get indices of liked movies
        liked_indices = movies_df.index[movies_df["movie_id"].isin(movie_ids)].tolist()
        # Compute cosine similarity using the TF-IDF matrix
        similarities = cosine_similarity(tfidf_matrix[liked_indices], tfidf_matrix)
        similarities = np.sum(similarities, axis=0) # aggregate similarities
        if similarities.max() > 0:
            similarities = similarities / similarities.max()

    # Collaborative Filtering: expected ratings
    test_set = [(user_id, mid, 0) for mid in movies_df["movie_id"]]
    expected_ratings = svd_model.test(test_set)
    expected_ratings_pred = np.array([pred.est for pred in expected_ratings])
    cf_scores = expected_ratings_pred / 5.0

    final_scores = weight_cb * similarities + weight_cf * cf_scores

    result_df = pd.DataFrame({
        "movie_id": movies_df["movie_id"],
        "title": movies_df["title"],
        "cb_score": similarities,
        "cf_score": cf_scores,
        "hybrid_score": final_scores
    })

    watched_movie_ids = ratings_df[ratings_df["user_id"] == user_id]["movie_id"].values
    result_df = result_df[~result_df["movie_id"].isin(watched_movie_ids)]

    return result_df.sort_values(by="hybrid_score", ascending=False).head(top_n)

# Test the function
hybrid_recommendation(2, movies, ratings, svd, genres)


,movie_id,title,cb_score,cf_score,hybrid_score
169,170,Cinema Paradiso (1988),1.000000,0.841492,0.889044
516,517,Manhattan (1979),1.000000,0.831705,0.882194
511,512,Wings of Desire (1987),1.000000,0.818453,0.872917
935,936,Brassed Off (1996),1.000000,0.790499,0.853349
356,357,One Flew Over the Cuckoo's Nest (1975),0.716023,0.908249,0.850581
480,481,"Apartment, The (1960)",0.865896,0.834811,0.844137
44,45,Eat Drink Man Woman (1994),0.865896,0.829260,0.840251
130,131,Breakfast at Tiffany's (1961),0.848089,0.833143,0.837627
777,778,Don Juan DeMarco (1995),1.000000,0.766330,0.836431
920,921,Farewell My Concubine (1993),0.848089,0.828407,0.834312


## Save Data and Models for Streamlit Deployment

In [22]:
import pickle

# Save models and data for Streamlit
with open('movies.pkl', 'wb') as f:
    pickle.dump(movies, f)

with open('ratings.pkl', 'wb') as f:
    pickle.dump(ratings, f)

with open('svd_model.pkl', 'wb') as f:
    pickle.dump(svd, f)

print("Models and Data saved successfully for Streamlit!")


Models and Data saved successfully for Streamlit!
